# Build the oceans shapefile

Generate a shapefile with all the oceans and seas grouped as follows:

| Ocean / Sea                              |
|:-----------------------------------------|
| Southern Ocean                           |
| Southern Atlantic Ocean                  |
| South Pacific Ocean                      |
| North Pacific Ocean                      |
| South China and Easter Archipelagic Seas |
| Indian Ocean                             |
| Mediterranean Region                     |
| Baltic Sea                               |
| North Atlantic Ocean                     |
| Arctic Ocean Caspian Sea                 |


### Input data:


**GOaS and SeaVox** @  https://www.marineregions.org/downloads.php:
   - Global Oceans and Seas, version 1 (Flanders Marine Institute, 2021) https://doi.org/10.14284/542 for the Oceans
   - Polygon dataset of the extent of water bodies from the SeaVoX Salt and Fresh Water Body Gazetteer (v19) (British Oceanographic Data Centre, 2023) https://doi.org/10.14284/590 for the Caspian Sea

### Input data license:

Global Oceans and Seas, version 1 (Flanders Marine Institute, 2021): CC BY 4.0 and Polygon dataset of the extent of water bodies from the SeaVoX Salt and Fresh Water Body Gazetteer (v19) (British Oceanographic Data Centre, 2023): CC BY 4.0

Giulia Cigna - giulia.cigna@polit.it<br>
Romain Thomas - romain.thomas@polito.it<br>
2026

In [1]:
import os
from dotenv import load_dotenv
import geopandas as gpd
import pandas as pd
import logging
from pathlib import Path
import chardet

## LOGGING

In [2]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(name)s - %(levelname)s - %(message)s",
    datefmt='%Y-%m-%d %H:%M:%S',
    force=True
)

## SETTINGS

In [3]:
if not os.path.exists(".env"):
    raise ValueError("You must create the '.env' file and set the values before running this notebook.")

load_dotenv()

True

## OUTPUT SETTINGS

In [4]:
# Get full shapefile path from environment
oceans_from_goas_seavox_path = os.getenv("OCEANS_FROM_GOAS_SEAVOX_PATH")
if oceans_from_goas_seavox_path is None:
    raise ValueError("OCEANS_FROM_GOAS_SEAVOX_PATH environment variable is not set")

# Create parent directory if it doesn't exist
Path(oceans_from_goas_seavox_path).parent.mkdir(parents=True, exist_ok=True)

logging.info(f"Output path: {oceans_from_goas_seavox_path}")


2026-03-13 18:00:24 - root - INFO - Output path: data/oceans_from_GOaS_SeaVoX/oceans_from_GOaS_SeaVoX.shp


## INPUT FILES

In [5]:
# GOaS shapefile
goas_oceans_path = os.getenv("GOAS_OCEANS_PATH")
if goas_oceans_path is None:
    raise ValueError("GOAS_OCEANS_PATH environment variable is not set")

# SeaVoX shapefile
seavox_seas_path = os.getenv("SEAVOX_SEAS_PATH")
if seavox_seas_path is None:
    raise ValueError("SEAVOX_SEAS_PATH environment variable is not set")

# CSV with regions ID
codes_id_path = os.getenv('CODES_ID_PATH')
if codes_id_path is None:
    raise ValueError("CODES_ID_PATH environment variable is not set")


## READING INPUT FILES

In [6]:
# goas
logging.info(f"Reading shapefile for GOaS from {goas_oceans_path}")
gdf_goas = gpd.read_file(goas_oceans_path)

# seavox
logging.info(f"Reading shapefile for SeaVox from {seavox_seas_path}")
gdf_seavox = gpd.read_file(seavox_seas_path)


2026-03-13 18:00:24 - root - INFO - Reading shapefile for GOaS from data/GOaS_v1_20211214/goas_v01.shp
2026-03-13 18:00:38 - root - INFO - Reading shapefile for SeaVox from data/SeaVoX_sea_areas_polygons_v19/SeaVoX_sea_areas_polygons_v19.shp


In [7]:
# regions ID
logging.info(f"Reading ids from {codes_id_path}")
with open(codes_id_path, "rb") as f:
    result = chardet.detect(f.read())

# from https://pandas.pydata.org/pandas-docs/stable/user_guide/io.html#na-values
na_vals = ['-1.#IND', '1.#QNAN', '1.#IND', '-1.#QNAN', '#N/A N/A', '#N/A', 'N/A', 'n/a', 'NA', '<NA>', '#NA', 'NULL', 'null', 'NaN', '-NaN', 'nan', '-nan', 'None', '']
# avoids errors with country code "NA":
na_vals.remove('NA')

codes_id = pd.read_csv(
    codes_id_path,
    encoding=result["encoding"],
    sep=None,
    engine="python",
    keep_default_na=False,
    na_values=na_vals
)


2026-03-13 18:00:58 - root - INFO - Reading ids from ../data/codes_id.csv


## ALIGNMENT CHECK

In [8]:
# Check if all the Reference Systems are the same
logging.info(f"GOaS reference system: {gdf_goas.crs}")
logging.info(f"SeaVox reference system: {gdf_seavox.crs}")

if gdf_goas.crs == gdf_seavox.crs:
    logging.info("Reference system aligned")
else:
    raise ValueError("Reference system not aligned")


2026-03-13 18:00:58 - root - INFO - GOaS reference system: EPSG:4326
2026-03-13 18:00:58 - root - INFO - SeaVox reference system: EPSG:4326
2026-03-13 18:00:58 - root - INFO - Reference system aligned


## METADATA


In [9]:
# Scheme definition of the final shape file

schema = {
    "ID": "str:10",
    "NAME": "str:50",
    "ISO3_CODE": "str:3",
    "ISO2_CODE": "str:2",
    "ISON_CODE": "int",
    "NUM_ID": "int",
    "OL_NAME": "str:50",
    "SOURCE": "str:50",
    "geometry": "MultiPolygon"
}


Mapping between original shape file columns and finale scheme

In [10]:
# Oceans
attr_goas = [c for c in gdf_goas.columns if c != "geometry"]

logging.info(f"Attributes in the Oceans shape file are: {attr_goas}")

2026-03-13 18:00:58 - root - INFO - Attributes in the Oceans shape file are: ['name', 'latitude', 'longitude', 'min_Y', 'min_X', 'max_Y', 'max_X', 'area_km2']


In [11]:
oceans_columns_map = {
    'name': "NAME"
}

In [12]:
# SeaVox
attr_seavox = [c for c in gdf_seavox.columns if c != "geometry"]

logging.info(f"Attributes in the SeaVox shape file are: {attr_seavox}")

2026-03-13 18:00:58 - root - INFO - Attributes in the SeaVox shape file are: ['LEVEL_4', 'LEVEL_3', 'LEVEL_2', 'LEVEL_1', 'REGION', 'SUB_REGION', 'SKOS_URL', 'MRGID_SR', 'MRGID_R', 'MRGID_L1', 'MRGID_L2', 'MRGID_L3', 'MRGID_L4', 'LAT', 'LONG', 'MINLAT', 'MINLONG', 'MAXLAT', 'MAXLONG', 'AREA_KM2']


In [13]:
seavox_columns_map = {
    'SUB_REGION': "NAME"
}

Mapping

In [14]:
# Mapping function

def prepare_gdf(gdf, col_map, fixed_values=None):
    gdf = gdf.copy()

    # Rename columns
    gdf = gdf.rename(columns=col_map)

    # Create missing columns with placeholders
    for col in schema:
        if col not in gdf.columns and col != "geometry":
            gdf[col] = None   # placeholder

    # Apply fixed values
    if fixed_values:
        for col, val in fixed_values.items():
            gdf[col] = val

    # Keep only target columns (ORDERED by schema)
    gdf = gdf[list(schema)]

    return gdf


In [15]:
gdf_goas = prepare_gdf(
    gdf_goas,
    oceans_columns_map
)

logging.info(f"Attributes in the modified Oceans shape file are: {gdf_goas.columns}")

gdf_seavox = prepare_gdf(
    gdf_seavox,
    seavox_columns_map
)

logging.info(f"Attributes in the modified SeaVox shape file are: {gdf_seavox.columns}")


if gdf_goas.columns.all() == gdf_seavox.columns.all():
    logging.info("Columns aligned")
else:
    logging.info("Columns system not aligned")

2026-03-13 18:00:58 - root - INFO - Attributes in the modified Oceans shape file are: Index(['ID', 'NAME', 'ISO3_CODE', 'ISO2_CODE', 'ISON_CODE', 'NUM_ID',
       'OL_NAME', 'SOURCE', 'geometry'],
      dtype='str')
2026-03-13 18:00:58 - root - INFO - Attributes in the modified SeaVox shape file are: Index(['ID', 'NAME', 'ISO3_CODE', 'ISO2_CODE', 'ISON_CODE', 'NUM_ID',
       'OL_NAME', 'SOURCE', 'geometry'],
      dtype='str')
2026-03-13 18:00:58 - root - INFO - Columns aligned


Add features to GOaS gdf

In [16]:
# Retrieve the Oceans ID
gdf_goas["ID"] = gdf_goas["NAME"].apply(
    lambda s: "OC_" + "".join(w[0].upper() for w in s.split())
)
# Change South China and Easter Archipelagic Seas ID
gdf_goas["ID"] = gdf_goas["ID"].replace(
    "OC_SCAEAS",
    "OC_SCS"
)

# Add source and duplicate the name
gdf_goas["SOURCE"] = "GOAS v1"
gdf_goas["OL_NAME"] = gdf_goas["NAME"]

Caspian Sea is not present in the Oceans file

In [17]:
casp = gdf_seavox[gdf_seavox["NAME"] == "CASPIAN SEA"]

# Add features to Caspian Sea
casp["NAME"] = "Caspian Sea"
casp["ID"] = "OC_CS"
casp["SOURCE"] = "SeaVox v19"
casp["OL_NAME"] = casp["NAME"]

## SAVING OUTPUT FILE

In [18]:
# Merge together the gdfs
gdf_final = gpd.GeoDataFrame(
    pd.concat([gdf_goas, casp], ignore_index=True),
    crs="EPSG:4326"
)

# Add the NUM_ID from the csv file
lookup = codes_id.set_index("ID")["NUM_ID"]
gdf_final["NUM_ID"] = gdf_final["ID"].map(lookup)
gdf_final["NUM_ID"] = gdf_final["NUM_ID"].astype(str).str.strip()
gdf_final["NUM_ID"] = pd.to_numeric(gdf_final["NUM_ID"], errors='coerce').astype('Int64')


In [19]:
# Save GeoDataFrame
gdf_final.to_file(oceans_from_goas_seavox_path)

logging.info(f"Saved Shapefile to: {oceans_from_goas_seavox_path}")

2026-03-13 18:00:58 - pyogrio._io - INFO - Created 11 records
2026-03-13 18:00:58 - root - INFO - Saved Shapefile to: data/oceans_from_GOaS_SeaVoX/oceans_from_GOaS_SeaVoX.shp
